# Lab 6.6 &mdash; Routing an Action to a Tool (safely)

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Look a tool up by name in a registry and run it with .invoke()
- Wrap the result as the observation the agent reads next
- Handle an unknown tool and a failing tool without crashing

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, routing and guardrails &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable.

**Reference:** [Module 6 slides &mdash; LangChain's core building blocks](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-06-06"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Once the model picks an **action name**, the framework must **route** it to the matching tool, run it
via `.invoke()`, and wrap the result as an **observation**. `create_agent` does this internally &mdash;
here you build the same safe dispatch by hand so you can debug it. Models hallucinate tool names, so
routing must fail **safely**: an unknown or failing tool returns a message, not a crash.

In [ ]:
from langchain_core.tools import tool

@tool
def weather(city: str) -> str:
    """Get the current weather for a city."""
    return "sunny, 24C"

print("a tool has .name and .invoke:", weather.name, "->", weather.invoke("tokyo"))

## Your Turn
Build a registry of real tools, then implement `dispatch`: find the tool, run it via `.invoke()`, and
handle the unknown/failing cases.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool

@tool
def calculator(expression: str) -> float:
    """Do exact arithmetic on an expression."""
    return safe_calc(expression)

@tool
def lookup(key: str) -> str:
    """Look up a known fact by key."""
    facts = {"capital of france": "Paris", "population of metropolis": "120000"}
    return facts.get(key.lower().strip(), "unknown")

REGISTRY = {t.name: t for t in [calculator, lookup]}

def dispatch(registry, name, arg):
    tool = ___   # TODO: get the tool by name (None if not registered)
    if tool is None:
        return {"ok": False, "observation": ___}   # TODO: a message naming the unknown tool
    try:
        return {"ok": True, "observation": ___}   # TODO: run the tool on arg via .invoke
    except Exception as e:
        return {"ok": False, "observation": "tool error: " + type(e).__name__}

try:
    print(dispatch(REGISTRY, 'calculator', '10/2'))
    print(dispatch(REGISTRY, 'lookup', 'capital of france'))
    print(dispatch(REGISTRY, 'no_such_tool', 'x'))
    print(dispatch(REGISTRY, 'calculator', 'not-math'))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("routes the calculator correctly", lambda: dispatch(REGISTRY, "calculator", "10/2")["observation"] == 5.0)
expect_true("routes the lookup correctly", lambda: dispatch(REGISTRY, "lookup", "capital of france")["observation"] == "Paris")
expect_true("ok is True on success", lambda: dispatch(REGISTRY, "lookup", "capital of france")["ok"] is True)
expect_true("an unknown tool is handled, naming it", lambda: dispatch(REGISTRY, "no_such_tool", "x")["ok"] is False and "no_such_tool" in dispatch(REGISTRY, "no_such_tool", "x")["observation"])
expect_true("a failing tool does not crash", lambda: dispatch(REGISTRY, "calculator", "not-math")["ok"] is False)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Routing turns a chosen action into a real observation -- safely. The split between the model deciding and the executor running is what makes agents debuggable.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>